# Rescoring of ECD, EID, ETciD and UVPD data using Oktoberfest

This notebook provides an overview of rescoring alternative fragmentation data in Oktoberfest. 

## 1. Import python packages

In [3]:
from pathlib import Path
from oktoberfest.runner import run_job
import json
import urllib.request
import os
import shutil
from tqdm import tqdm

## 2. Installations

### Percolator:
- To install percolator on windows download suitable file from: https://github.com/percolator/percolator/releases
Windows:
- Run the downloaded file while running the setup make sure to select "add percolator to the system PATH for all users" when asked.
Linux:
- You can define path as:
```python
os.environ["PATH"] += os.pathsep + "/full/path/to/percolator/bin"
```


### Oktoberfest:
- Oktoberfest currently supports Python versions 3.10, 3.11 and 3.12. Support for 3.13 will be added in the near future.
- Install oktoberfest using pip install oktoberfest

## 3. Download files from zenodo required for rescoring task

The data used in this tutorial is available through a public Zenodo record.
The dataset is approximately 1.7 GB in size and for each fragmentation (ECD, EID, ETciD, HCD, UVPD) technique includes:

- 1 sample mzML file used for rescoring peptides.
- FragPipe results of mzML.


### Get the current directory and set the file name

In [ ]:
# Define which technique you want to download
fragmentation_method = "ECD"
download_dir = Path.cwd()
download_file = download_dir / f"{fragmentation_method}.tar"
url = f"https://zenodo.org/records/18199550/files/{fragmentation_method}.tar?download=1"

# If you want to download all datasets
# url = "https://zenodo.org/api/records/18199550/files-archive"

# set download to False if you already have the file and don"t want to download again in the next step
download = True

if download:
    with tqdm(unit="B", total=639600000, unit_scale=True, unit_divisor=1000, miniters=1, desc=url.split("/")[-1]) as t:
        urllib.request.urlretrieve(
            url=url,
            filename=download_file,
            reporthook=lambda blocks, block_size, _: t.update(blocks * block_size - t.n)
        )
    shutil.unpack_archive(download_file, download_dir)

    # For all datasets
    # for method in ["ECD", "EID", "ETciD", "HCD", "UVPD"]:
    #   download_file = download_dir / f"{fragmentation_method}.tar"
    #   shutil.unpack_archive(download_file, download_dir)

## 4. Rescoring


**Important**: The custom ions in the Prosit-MultiFrag and Oktoberfest defined as following
```python
a:  a
A:  a+1
b:  b
c:  c-1
C:  c
x:  x
X:  x+1
y:  y
z:  z
Z:  z+1
```

### Config file

In [ ]:
fragmentation_method = "ECD"

In [ ]:
rescoring_config = {
    "type": "Rescoring",
    "tag": "",
    "output": "./out",
    "inputs": {
        "search_results": f"{download_dir / fragmentation_method / 'fragpipe_search'}",
        "search_results_type": "Msfragger",
        "spectra": f"{download_dir / fragmentation_method / 'mzml'}",
        "spectra_type": "mzML"
    },
    "models": {
        "irt": "Prosit_2019_irt",
        "intensity": "Prosit_2025_intensity_MultiFrag"
    },
    "prediction_server": "koina.wilhelmlab.org:443",
    "numThreads": 1,
    "fdr_estimation_method": "percolator", # ensure percolator is installed on your system
    "regressionMethod": "spline",
    "ssl": True,
    "massTolerance": 20,
    "p_window": 1.2,
    "unitMassTolerance": "ppm",
    "fragmentation_method": f"{fragmentation_method}",
    "featured_ions": [ # define which ion types will be considered in rescoring
        "C", # c
        "z" # z
        ]
} 

In [29]:
with open('./rescoring_config.json', 'w') as fp:
    json.dump(rescoring_config, fp)

### Run rescoring 

In [30]:
run_job("rescoring_config.json")

2025-12-19 15:53:01,969 - INFO - oktoberfest.utils.config::read Reading configuration from rescoring_config.json
2025-12-19 15:53:02,033 - INFO - oktoberfest.runner::run_job Oktoberfest version 0.10.0
Copyright 2025, Wilhelmlab at Technical University of Munich
2025-12-19 15:53:02,034 - INFO - oktoberfest.runner::run_job Job executed with the following config:
2025-12-19 15:53:02,036 - INFO - oktoberfest.runner::run_job {
    "type": "Rescoring",
    "tag": "",
    "output": "./out",
    "inputs": {
        "search_results": "/cmnfs/proj/prosit_multifrag/oktoberfest_runs/oktoberfest_tutorial/input_files/ECD/fragpipe_search",
        "search_results_type": "Msfragger",
        "spectra": "/cmnfs/proj/prosit_multifrag/oktoberfest_runs/oktoberfest_tutorial/input_files/ECD/mzml",
        "spectra_type": "mzML"
    },
    "models": {
        "irt": "Prosit_2019_irt",
        "intensity": "Prosit_2025_intensity_MultiFrag"
    },
    "prediction_server": "koina.wilhelmlab.org:443",
    "numTh

Waiting for tasks to complete:   0%|          | 0/1 [00:00<?, ?it/s]

2025-12-19 15:53:02,114 - INFO - oktoberfest.utils.process_step::is_done Skipping ce_calib.EXPLO_NGL_YD1_241122_Expi_tryps_ECD50ms_900kHz_4p6A_frac10_20 step because out/proc/ce_calib.EXPLO_NGL_YD1_241122_Expi_tryps_ECD50ms_900kHz_4p6A_frac10_20.done was found.


Waiting for tasks to complete:   0%|          | 0/1 [00:00<?, ?it/s]

2025-12-19 15:53:06,175 - INFO - oktoberfest.utils.process_step::is_done Skipping ce_calib.EXPLO_NGL_YD1_241122_Expi_tryps_ECD50ms_900kHz_4p6A_frac10_20 step because out/proc/ce_calib.EXPLO_NGL_YD1_241122_Expi_tryps_ECD50ms_900kHz_4p6A_frac10_20.done was found.
2025-12-19 15:53:07,338 - INFO - oktoberfest.utils.process_step::is_done Skipping calculate_features.EXPLO_NGL_YD1_241122_Expi_tryps_ECD50ms_900kHz_4p6A_frac10_20 step because out/proc/calculate_features.EXPLO_NGL_YD1_241122_Expi_tryps_ECD50ms_900kHz_4p6A_frac10_20.done was found.


Waiting for tasks to complete: 100%|██████████| 1/1 [00:01<00:00,  1.17s/it]

2025-12-19 15:53:07,372 - INFO - oktoberfest.utils.process_step::is_done Skipping percolator_prepare_tab_original step because out/proc/percolator_prepare_tab_original.done was found.
2025-12-19 15:53:07,375 - INFO - oktoberfest.utils.process_step::is_done Skipping percolator_prepare_tab_prosit step because out/proc/percolator_prepare_tab_prosit.done was found.
2025-12-19 15:53:07,377 - INFO - oktoberfest.rescore.rescore::rescore_with_percolator Starting percolator with command percolator --weights out/results/percolator/original.percolator.weights.csv                         --num-threads 3                         --subset-max-train 500000                         --post-processing-tdc                         --testFDR 0.01                         --trainFDR 0.01                         --results-psms out/results/percolator/original.percolator.psms.txt                         --decoy-results-psms out/results/percolator/original.percolator.decoy.psms.txt                         --result

2025-12-19 15:53:13,426 - INFO - oktoberfest.rescore.rescore::rescore_with_percolator Finished rescoring using percolator.
2025-12-19 15:53:13,429 - INFO - oktoberfest.runner::_rescore False
2025-12-19 15:53:13,430 - INFO - oktoberfest.rescore.rescore::rescore_with_percolator Starting percolator with command percolator --weights out/results/percolator/rescore.percolator.weights.csv                         --num-threads 3                         --subset-max-train 500000                         --post-processing-tdc                         --testFDR 0.01                         --trainFDR 0.01                         --results-psms out/results/percolator/rescore.percolator.psms.txt                         --decoy-results-psms out/results/percolator/rescore.percolator.decoy.psms.txt                         --results-peptides out/results/percolator/rescore.percolator.peptides.txt                         --decoy-results-peptides out/results/percolator/rescore.percolator.decoy.peptides.txt 